In [1]:
#Creamos las variables de memoria de embalse y de lluvia reciente acumulada, para que cada semana lleve su pasada como columna.

import pandas as pd

df = pd.read_parquet("../data/processed/dataset_semanal.parquet")

df["cambio_1s"] = df["volumen"].diff(1)
df["cambio_4s"] = df["volumen"].diff(4)

df["lluvia_lag1"] = df["lluvia"].shift(1)
for n in [4, 12, 26]:
    df[f"lluvia_{n}s"] = df["lluvia"].rolling(n).sum()

df.loc["1994-02-01":"1994-03-01"]

,lluvia,tmed,volumen,capacidad,pct_llenado,cambio_1s,cambio_4s,lluvia_lag1,lluvia_4s,lluvia_12s,lluvia_26s
fecha,,,,,,,,,,,
1994-02-01,0.02,10.557143,1020.0,5878.0,17.352841,NaN,NaN,0.04,53.58,64.44,200.60
1994-02-08,13.12,7.300000,1030.0,5878.0,17.522967,10.0,NaN,0.02,13.22,77.30,213.32
1994-02-15,10.52,9.322857,1047.0,5878.0,17.812181,17.0,NaN,13.12,23.70,86.84,223.84
1994-02-22,5.00,10.871429,1070.0,5878.0,18.203471,23.0,NaN,10.52,28.66,88.06,228.84
1994-03-01,30.52,12.194286,1070.0,5878.0,18.203471,0.0,50.0,5.00,59.16,118.58,259.32


In [2]:
#Añadimos estacionalidad, el estado del suelo (lluvia desde el inicio del año hidrologico), la temperatura reciente y los objetivos 
#a 4 y 8 semanas.

df["semana"] = df.index.isocalendar().week.astype(int)

año_hidro = df.index.year + (df.index.month >= 10)
df["lluvia_año_hidro"] = df.groupby(año_hidro)["lluvia"].cumsum()

df["tmed_4s"] = df["tmed"].rolling(4).mean()

df["objetivo_4s"] = df["volumen"].shift(-4) - df["volumen"]
df["objetivo_8s"] = df["volumen"].shift(-8) - df["volumen"]

df[["semana", "lluvia_año_hidro", "tmed_4s", "volumen", "objetivo_4s", "objetivo_8s"]].tail(10)

,semana,lluvia_año_hidro,tmed_4s,volumen,objetivo_4s,objetivo_8s
fecha,,,,,,
2026-08-18,34,642.04,29.837738,4295.0,-226.0,-371.0
2026-08-25,35,643.90,28.931250,4232.0,-209.0,-320.0
2026-09-01,36,644.26,27.798393,4170.0,-194.0,NaN
2026-09-08,37,644.36,28.063393,4113.0,-170.0,NaN
2026-09-15,38,644.36,27.122321,4069.0,-145.0,NaN
2026-09-22,39,644.36,27.134286,4023.0,-111.0,NaN
2026-09-29,40,646.32,27.352143,3976.0,NaN,NaN
2026-10-06,41,10.84,25.766429,3943.0,NaN,NaN
2026-10-13,42,10.84,24.366071,3924.0,NaN,NaN


In [3]:
#Definimos la lista de variables del modelo, quitamos las semanas del 1993 sin embalse y guardamos el dataset de features para el modelado.

FEATURES = [
    "volumen", "pct_llenado", "cambio_1s", "cambio_4s",
    "lluvia", "lluvia_lag1", "lluvia_4s", "lluvia_12s", "lluvia_26s",
    "lluvia_año_hidro", "semana", "tmed", "tmed_4s",
]
OBJETIVOS = ["objetivo_4s", "objetivo_8s"]

df = df[df["volumen"].notna()]
df[FEATURES + OBJETIVOS].to_parquet("../data/processed/features.parquet")

print(df.shape)
print(df[FEATURES].isna().sum()[lambda s: s > 0])

(1708, 16)
cambio_1s    1
cambio_4s    4
dtype: int64
